# Research Notebook for PISA question and answers on different languages

## Libraries

In [12]:
import os, getpass

In [13]:
from openai import OpenAI

In [35]:
# --- 0) Setup: imports & config
import os, re, time, json, math, random
from typing import Dict, List, Tuple, Any, Optional
import pandas as pd

## Configuration

In [18]:
BASE_URL = "https://ri-delta.ai/"  # <- replace with your portal URL root
MODEL_GPT = "gpt-5"                    # <- replace with exact model id if different
MODEL_CLAUDE = "claude-sonnet-4"
MODEL_GEMINI = "gemini-2.5-pro"

In [15]:
client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=BASE_URL + "/api"    # e.g., https://llm.company.com/v1
)

In [26]:
SHEET_ID = "1QVPzB7uMwqJ6jCsHkwIILnXvDQIycpqkcV3bkiDpzyQ"
WORKSHEET_NAME = "dataset"  # change if needed
RESULTS_CSV = "llm_eval_results.csv"
SAMPLE_PER_LANGUAGE = 1     # 1 per language
MAX_LANGUAGES = 10          # 10 languages total
# LLM_TEMPERATURE = 0.2
# LLM_MAX_TOKENS = 256
SEED = 42

random.seed(SEED)

## Load data from Google Sheet

In [25]:
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={WORKSHEET_NAME}"
try:
    df = pd.read_csv(csv_url)
except Exception as e:
    raise RuntimeError(
        "Failed to read the Google Sheet via CSV export. "
        "Make sure the sheet is shared as 'Anyone with the link can view', "
        f"ID is correct, and tab name matches. Underlying error: {e}"
    )

expected_cols = {
    "qid","language","question","context","options","gold",
    "answer_type","category","difficulty","rationale","source"
}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"Your sheet is missing columns: {sorted(missing)}")

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105 entries, 0 to 104
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   qid          105 non-null    object
 1   language     105 non-null    object
 2   question     105 non-null    object
 3   context      105 non-null    object
 4   options      105 non-null    object
 5   gold         105 non-null    object
 6   answer_type  105 non-null    object
 7   category     105 non-null    object
 8   difficulty   105 non-null    object
 9   rationale    44 non-null     object
 10  source       105 non-null    object
dtypes: object(11)
memory usage: 9.2+ KB


## Normalize & sample

In [29]:
df["language"] = df["language"].astype(str).str.strip()
lang_groups = []
for lang, sub in df.groupby("language", sort=True):
    lang_groups.append(sub.iloc[:SAMPLE_PER_LANGUAGE])

In [30]:
sampled = pd.concat(lang_groups, ignore_index=True).iloc[:MAX_LANGUAGES]
if sampled.empty:
    raise ValueError("No rows selected. Check your data.")

In [31]:
print(f"Selected {len(sampled)} rows across {sampled['language'].nunique()} languages.")
display(sampled[["qid","language","question","gold"]])

Selected 10 rows across 10 languages.


,qid,language,question,gold
0,q0006,Albanian,Çfarë përmendin shkencëtarët në artikull për t...,B
1,q0001,Arabic,إذا قررتْ دانا شراء السيارة (د) وباعتها بعد ثل...,C
2,q0001,Chinese,如果譚雅決定購買汽車 D 並於三年後在保持良好狀態下轉售，那麼這輛汽車的大約轉售價格將是多少...,C
3,q0001,Czech,Jaká bude přibližná prodejní cena auta (v zede...,C
4,q0001,English,If Tania decides to buy car D and resell it af...,C
5,q0001,Georgian,"თუ ქეთი გადაწყვეტს, იყიდოს მანქანა D და გაყიდო...",C
6,q0001,Kazakh,Егер Тоғжан Г автокөлігін сатып алуды шешіп жә...,C
7,q0001,Mongolian,"Хэрэв Туяа Г машиныг худалдан аваад, хамгийн с...",C
8,q0001,Russian,Если Таня решит купить автомобиль D и перепрод...,C
9,q0001,Thai,ถ้าทิชาตัดสินใจซื้อรถยนต์ D และขายรถยนต์คันนี้...,C


## Parse options

In [32]:
def parse_options(raw: str) -> dict:
    """
    Parse options when stored as a JSON list of labeled strings, e.g.:
        ["A) 1575", "B) 8925", "C) 9000", "D) 9975"]

    Returns a dict like:
        {"A": "1575", "B": "8925", "C": "9000", "D": "9975"}
    """
    if pd.isna(raw):
        raise ValueError("Options are empty")

    try:
        # Try to load as JSON list
        items = json.loads(raw)
        if not isinstance(items, list):
            raise ValueError("Expected a list of options")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON format for options: {e}")

    options = {}
    for item in items:
        if not isinstance(item, str):
            raise ValueError(f"Option is not a string: {item}")
        # Match patterns like "A) text", "B. text", or "C: text"
        match = re.match(r"^\s*([A-Z])[\)\.\:]\s*(.+)$", item.strip())
        if match:
            label, text = match.groups()
            options[label.upper()] = text.strip()
        else:
            # Fallback: assign next available letter automatically
            next_label = chr(ord('A') + len(options))
            options[next_label] = item.strip()

    return options


## Build prompt

In [101]:
def build_prompt(row: pd.Series, options: Dict[str,str]) -> str:
    """
    Builds a language-agnostic but context-aware prompt for MCQ.
    """
    options_block = "\n".join([f"{k}. {v}" for k,v in options.items()])
    context_block = f"\nContext:\n{row['context']}\n" if pd.notna(row["context"]) and str(row["context"]).strip() else ""
    # You can adapt language instruction if you want the rationale in the same language:
    lang = str(row["language"]).strip()

    return (
        # f"You are answering a multiple-choice question. "
        # f"Return ONLY the chosen option letter (A, B, C, ...). "
        f"{context_block}\n"
        f"{row['question']}\n"
        f"{options_block}\n"
        # f"\nReply format STRICTLY:\n" 
        # f"<LETTER>\n" 
    )

In [66]:
answer_letter_regex = re.compile(r"<\s*([A-Z])\s*>")

def extract_letter(text: str, valid_letters: List[str]) -> str:
    """
    Extract the first single-letter A-Z token that is in valid_letters.
    """
    if not text:
        return ""
    # First line is the letter per our format; but still be defensive:
    first_line = text.strip().splitlines()[0].strip()
    # If first line is a single valid letter, use it
    if len(first_line) == 1 and first_line.upper() in valid_letters:
        return first_line.upper()
    # Else find any A-Z token
    m = answer_letter_regex.search(text.upper())
    if m and m.group(1) in valid_letters:
        return m.group(1)
    return ""

## LLM call wrapper

In [102]:
def llm_completion(
    prompt: str,
    model: Optional[str] = None,
    # temperature: float = 0.2,
    # max_tokens: int = 256,
    system_prompt: str = "You are a helpful assistant that answers multiple-choice questions. Reply format: <LETTER>.",
    retries: int = 2,
    backoff_seconds: float = 1.5,
    **kwargs,
) -> str:
    """
    Call an LLM via chat.completions and return text.
    - `prompt`: user content (string)
    - `model`: overrides global MODEL_NAME if provided
    - `temperature`, `max_tokens`: usual decoding controls
    - `system_prompt`: system role content
    - `retries`: retry on transient errors
    - `backoff_seconds`: base backoff between retries
    - `**kwargs`: forwarded to client.chat.completions.create (e.g., stop, seed)
    """
    mdl = model or MODEL_CLAUDE
    last_err = None

    for attempt in range(retries + 1):
        try:
            # Build arguments dynamically — include only if provided
            call_args = dict(
                model=mdl,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
                stream=False
            )

            # Add optional params only if explicitly set
            if "temperature" in kwargs and kwargs["temperature"] is not None:
                call_args["temperature"] = kwargs["temperature"]
            if "max_tokens" in kwargs and kwargs["max_tokens"] is not None:
                call_args["max_tokens"] = kwargs["max_tokens"]

            # Merge other kwargs (e.g., stop, seed, etc.)
            for k, v in kwargs.items():
                if k not in call_args:
                    call_args[k] = v

            # Actual model call
            resp = client.chat.completions.create(**call_args)
            return (resp.choices[0].message.content or "").strip()

        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(backoff_seconds * (attempt + 1))
            else:
                raise

## Evaluation loop

In [64]:
def eval_rows(rows: pd.DataFrame, model_name: str = None, sleep_s: float = 0.0, retries: int = 2) -> pd.DataFrame:
    results = []
    for i, row in rows.iterrows():
        qid = row["qid"]
        lang = row["language"]
        gold = str(row["gold"]).strip().upper()

        # Parse options
        try:
            opts = parse_options(row["options"])
        except Exception as e:
            results.append({
                "qid": qid, "language": lang, "pred": "",
                "gold": gold, "is_correct": False,
                "error": f"OptionsParseError: {e}", "raw": ""
            })
            continue

        valid_letters = sorted(list(opts.keys()))
        prompt = build_prompt(row, opts)

        # call model with simple retry
        raw = ""
        err = ""
        for attempt in range(retries + 1):
            try:
                raw = llm_completion(prompt, model = model_name)
                break
            except Exception as e:
                err = f"{type(e).__name__}: {e}"
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                else:
                    raw = ""
        pred = extract_letter(raw, valid_letters)
        is_correct = (pred == gold)

        results.append({
            "qid": qid,
            "language": lang,
            "pred": pred,
            "gold": gold,
            "is_correct": bool(is_correct),
            "error": err,
            "raw": raw,
            "question": row["question"],
            "options_json": json.dumps(opts, ensure_ascii=False),
        })
        if sleep_s > 0:
            time.sleep(sleep_s)
    return pd.DataFrame(results)

## Execution and results

In [103]:
MODELS_TO_TEST = [
    ("GPT", MODEL_GPT),        # (tag, model_name_for_API)
    ("Claude", MODEL_CLAUDE),
    ("Gemini", MODEL_GEMINI)
]

In [104]:
filtered_df = df[df["language"] == "Albanian"]

In [105]:
all_results = []
for tag, model_name in MODELS_TO_TEST:
    print(f"\nEvaluating {tag} -> {model_name}")
    df_model = eval_rows(filtered_df, model_name=model_name) # sampled or df or filtered_df
    df_model["model_tag"] = tag
    df_model["model_name"] = model_name
    all_results.append(df_model)


Evaluating GPT -> gpt-5

Evaluating Claude -> claude-sonnet-4

Evaluating Gemini -> gemini-2.5-pro


In [106]:
res_df = pd.concat(all_results, ignore_index=True)

In [107]:
overall_by_model = (
    res_df.groupby(["model_tag","model_name"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by model:")
display(overall_by_model)


Overall accuracy by model:


,model_tag,model_name,accuracy
0,Claude,claude-sonnet-4,0.8
1,GPT,gpt-5,0.8
2,Gemini,gemini-2.5-pro,0.5


In [108]:
overall_by_lang = (
    res_df.groupby(["language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values("accuracy", ascending=False)
)
print("\nOverall accuracy by lang:")
display(overall_by_lang)


Overall accuracy by lang:


,language,accuracy
0,Albanian,0.7


In [109]:
by_model_lang = (
    res_df.groupby(["model_tag","model_name","language"])["is_correct"]
    .mean()
    .reset_index()
    .rename(columns={"is_correct":"accuracy"})
    .sort_values(["model_tag","language"])
)
print("\nAccuracy by model & language:")
display(by_model_lang)


Accuracy by model & language:


,model_tag,model_name,language,accuracy
0,Claude,claude-sonnet-4,Albanian,0.8
1,GPT,gpt-5,Albanian,0.8
2,Gemini,gemini-2.5-pro,Albanian,0.5


In [110]:
def safe_acc(s):
    return float('nan') if s.empty else s.mean()
overall_acc = safe_acc(res_df["is_correct"])
print(f"\nCombined overall accuracy on {len(res_df)} items: {overall_acc:.3f}")


Combined overall accuracy on 30 items: 0.700


In [111]:
# One combined file + one per-model file
RESULTS_COMBINED_CSV = "llm_eval_results__combined.csv"
res_df.to_csv(RESULTS_COMBINED_CSV, index=False)
print(f"Saved combined results to: {RESULTS_COMBINED_CSV}")

for tag, _ in MODELS_TO_TEST:
    out_path = f"llm_eval_results__{tag}.csv"
    res_df.query("model_tag == @tag").to_csv(out_path, index=False)
    print(f"Saved {tag} results to: {out_path}")

# (Optional) quick peek
display(res_df.head())

Saved combined results to: llm_eval_results__combined.csv
Saved GPT results to: llm_eval_results__GPT.csv
Saved Claude results to: llm_eval_results__Claude.csv
Saved Gemini results to: llm_eval_results__Gemini.csv


,qid,language,pred,gold,is_correct,error,raw,question,options_json,model_tag,model_name
0,q0006,Albanian,B,B,True,,B,Çfarë përmendin shkencëtarët në artikull për t...,"{""A"": ""Njerëzit u vendosën në Rapa Nui qindra ...",GPT,gpt-5
1,q0007,Albanian,D,D,True,,D,Çfarë provash paraqesin Carl Lipo dhe Terry Hu...,"{""A"": ""Minjtë arritën në ishull me kanoet e nj...",GPT,gpt-5
2,q0008,Albanian,B,B,True,,B,"Sipas IDFA, me cilën thënie bien dakord organi...","{""A"": ""Konsumi i qumështit dhe produktet e tij...",GPT,gpt-5
3,q0009,Albanian,D,D,True,,D,Cili është qëllimi kryesor i këtij teksti?,"{""A"": ""Të argumentojë se produktet e qumështit...",GPT,gpt-5
4,q0010,Albanian,,A,False,,A.,A është thënia fakt apo opinion?\nStudimet e f...,"{""A"": ""Opinion, Fakt, Fakt, Opinion."", ""B"": ""F...",GPT,gpt-5
